In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from tqdm import tqdm

In [3]:
# -------------------------------
# 1. Load Pretrained SSL Encoder
# -------------------------------
base_encoder = models.resnet18(weights=None)
simclr_encoder = SimCLR(base_encoder)
simclr_encoder.load_state_dict(torch.load("ssl_encoder.pth", map_location=device))

# Freeze most layers except the last few for fine-tuning
for name, param in simclr_encoder.encoder.named_parameters():
    param.requires_grad = "layer4" in name or "fc" in name

NameError: name 'SimCLR' is not defined

In [ ]:
# -------------------------------
# 2. Define Fine-tune Classifier
# -------------------------------
class FineTunedModel(nn.Module):
    def __init__(self, encoder, num_classes=2):
        super().__init__()
        self.encoder = encoder.encoder  # use only the feature extractor
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        feats = self.encoder(x)
        return self.classifier(feats)

model = FineTunedModel(simclr_encoder).to(device)

In [ ]:
# -------------------------------
# 3. Setup Loss, Optimizer, Scheduler
# -------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
# -------------------------------
# 4. Fine-tuning Loop
# -------------------------------
def fine_tune(model, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0
        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
        acc = 100. * correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = outputs.max(1)
                val_correct += preds.eq(labels).sum().item()
                val_total += labels.size(0)
        val_acc = 100. * val_correct / val_total
        scheduler.step()

        print(f"Epoch [{epoch+1}/{epochs}] - Train Acc: {acc:.2f}% | Val Acc: {val_acc:.2f}%")

    return model

In [ ]:
# -------------------------------
# 5. Run Fine-tuning
# -------------------------------
fine_tuned_model = fine_tune(model, train_loader, val_loader, epochs=10)

# -------------------------------
# 6. Save Fine-tuned Model
# -------------------------------
torch.save(fine_tuned_model.state_dict(), "fine_tuned_model.pth")
print("✅ Fine-tuning complete. Model saved as fine_tuned_model.pth")

In [ ]:
# Fine Tuned PKL
import pickle
import torch
import numpy as np

# Switch to evaluation mode
fine_tuned_model.eval()
features, labels = [], []

with torch.no_grad():
    for imgs, lbls in val_loader:  # or train_loader, depending on what you want analyzed
        imgs = imgs.to(device)
        feats = fine_tuned_model.encoder(imgs)  # Extract fine-tuned encoder features
        features.append(feats.cpu().numpy())
        labels.append(lbls.numpy())

# Concatenate
X = np.concatenate(features)
y = np.concatenate(labels)

# Save as .pkl for analysis
with open("fine_tuned_features.pkl", "wb") as f:
    pickle.dump({"features": X, "labels": y}, f)

print("✅ Fine-tuned features saved as fine_tuned_features.pkl")
